# Apollo 11 · Ask a question and inspect the evidence

This notebook compiles the pinned Apollo 11 SysML v2 model and runs **Text
Hybrid, One-Hop and Routed GraphRAG** on the question you enter. It builds
the projection and indexes once per kernel. Change `question` in step 3
and rerun from there to retrieve and pack new evidence.

The notebook uses the public simplified candidate, including scoped
lookup, the fallback router and fixed clarification responses. See the
[setup guide](README.md) for Java, compiler, model and tokenizer inputs.
An optional final cell sends the exact prompt to a pinned local Llama via
Ollama. Retrieval and prompt inspection work offline without Ollama.

## 1. Point to the local inputs

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

here = Path.cwd().resolve()
repository = next((p for p in (here, *here.parents)
                   if (p / 'sysml_graphrag/policy.py').is_file()), None)
if repository is None:
    raise RuntimeError('Start Jupyter inside the simplified repository.')
sys.path.insert(0, str(repository))

from examples.apollo.compare import ApolloComparison
from examples.apollo.demo import QUESTION, compile_model, generate_local, project_model

source = Path(os.environ.get('SYSML_APOLLO_SOURCE', '<Apollo source checkout>'))
pilot = Path(os.environ.get('SYSML_PILOT_DIR', '<extracted SysML Pilot sysml directory>'))
encoder = Path(os.environ.get('SYSML_ENCODER_DIR', '<local MiniLM snapshot>'))
tokenizer = Path(os.environ.get('SYSML_TOKENIZER_DIR', '<local Llama 3.1 chat tokenizer>'))
for label, folder in [('Apollo model', source), ('SysML Pilot', pilot),
                      ('MiniLM encoder', encoder), ('Llama tokenizer', tokenizer)]:
    if not folder.is_dir():
        raise FileNotFoundError(f'Set the {label} directory in this cell: {folder}')

## 2. Compile, project and index

The Java observer compiles the pinned model and writes the export. The
Python projector builds the graph. Text Hybrid indexes declarations
from the same staged source. One-Hop indexes graph neighborhoods. Both
indexes use local MiniLM and are reused on subsequent questions.

In [ ]:
scratch_parent = os.environ.get('SYSML_DEMO_SCRATCH')
out = Path(tempfile.mkdtemp(prefix='sysml-apollo-', dir=scratch_parent)) / 'run'
export = compile_model(source, pilot, out)
graph = project_model(export)
print(f'{len(graph):,} projected nodes, {graph.number_of_edges():,} edges')
print(f'Compiler log, export and projection: {out}')

comparison = ApolloComparison(graph, out / 'sources', encoder, tokenizer)
print('Ready for questions.')

## 3. Ask or change a question

Each method gets its own pack with a 4,096-token evidence cap, an
8,192-token context and a 512-token output reserve. Start with the
interface question, then try a question about declared parts or the
ambiguous name `engine1`. Free-form questions have no reference card or
accuracy score. Coverage depends on the Apollo model and the retrieval
policy.

In [ ]:
question = QUESTION
# question = 'Which parts does LaunchSite declare directly, and what type is each?'
# question = 'What is the declared type of engine1?'

results = comparison.prepare(question)
print(next(iter(results.values()))['query'])
for method, row in results.items():
    route = row['router_result']
    print(f"{method}: {route['route'] if route else 'ranked search'}"
          f" | {len(row['packed_units'])} packed units"
          f" | {row['prompt_fit']['input_tokens_to_send']} input tokens to send")
if decision := results['graph_routed'].get('candidate_decision'):
    print('Routed selection:', decision['reason'])

## 4. Inspect the ranked candidates and what reached the prompt

The first three candidate IDs point to source declarations or graph
records. Packed evidence below is the full context supplied to the answer
model. Compare source locators and the requested relationship roles across
methods. A fixed router response ends that method before generation.

In [ ]:
for method, row in results.items():
    print(f'\n{method}')
    if not row['generation_planned']:
        print(row['prompt_fit']['fixed_response'])
        continue
    packed_ids = set(row['packed']['selected_unit_ids'])
    for unit in row['ranked_units'][:3]:
        label = 'packed' if unit['id'] in packed_ids else 'omitted'
        print(f"  {label}: {unit['id']}")
    print(row['packed']['evidence'])

## 5. Optional local generation

Set `generate` to `True` with Ollama running locally and the pinned model
installed. Select which methods to show. For a fixed response the same
function returns the router's text without calling Ollama.

In [ ]:
generate = False
answer_methods = ('graph_routed',)  # Or tuple(results) for all three methods.
if generate:
    for method in answer_methods:
        row = results[method]
        if row['query'] != question.strip():
            raise RuntimeError('Run step 3 for the current question first.')
        response = generate_local(row)
        print(f'\n{method}: {response["answer"]}')
        print({key: value for key, value in response.items() if key != 'answer'})